# Assignment 3. Обработка естественного языка (NLP)

In [ ]:
%pip install -q datasets "transformers<5" sentencepiece rouge_score fasttext-predict networkx

In [ ]:
import re
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("vader_lexicon", quiet=True)

import sklearn, transformers, torch, datasets
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| nltk", nltk.__version__)
print("transformers", transformers.__version__, "| torch", torch.__version__, "| datasets", datasets.__version__)

# Задание 1. Анализ тональности текста

Три подхода: словарь (VADER), TF-IDF + Logistic Regression, готовая Transformer-модель (RoBERTa).

### Шаг 1. Подготовка набора данных

- Выбрать размеченный набор данных для анализа тональности, содержащий три класса (например, подходящий набор отзывов Amazon, TweetEval Sentiment или другой датасет с отрицательными, нейтральными и положительными метками).
- Использовать не менее 3 000 размеченных текстов, включая минимум 500 примеров каждого класса.
- Проанализировать распределение классов тональности.
- Удалить дублирующиеся записи, которые могут привести к утечке данных.
- Разделить набор данных на обучающую (70%), валидационную (15%) и тестовую (15%) выборки, применяя стратификацию при необходимости.

**Датасет:** TweetEval Sentiment (`cardiffnlp/tweet_eval`, конфиг `sentiment`, Hugging Face), test-часть. Англоязычные твиты, 3 класса: 0 = negative, 1 = neutral, 2 = positive.

In [ ]:
#1
from datasets import load_dataset

dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")
full = dataset["test"].to_pandas()

#3
print("Всего твитов в test-части:", len(full))
print("Распределение классов в исходных данных:")
print(full["label"].value_counts().sort_index())

In [ ]:
#4
print("Дубликатов:", full.duplicated(subset="text").sum())
full = full.drop_duplicates(subset="text").reset_index(drop=True)

#2
df = full.groupby("label").sample(n=1000, random_state=SEED).reset_index(drop=True)

#3
counts = df["label"].value_counts().sort_index()
print("Всего текстов:", len(df))
print(pd.DataFrame({"текстов": counts, "доля, %": (counts / len(df) * 100).round(1)}))

In [ ]:
#5
train_df, rest_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["label"])
val_df, test_df = train_test_split(rest_df, test_size=0.50, random_state=SEED, stratify=rest_df["label"])

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

print("Обучающая:", len(train_df))
print("Валидационная:", len(val_df))
print("Тестовая:", len(test_df))

**Распределение классов.** В исходной test-части классы несбалансированы: нейтральных почти в 2,5 раза больше, чем позитивных. Взято по 1000 твитов каждого класса — в итоговом наборе классы равны. Дубликатов нет.

### Шаг 2. Предварительная обработка текста

Реализовать и сравнить подходящие методы предварительной обработки:

- Токенизация (Tokenization).
- Приведение текста к нижнему регистру.
- Удаление несущественных символов.
- Удаление стоп-слов (Stopword Removal), если это целесообразно.
- Лемматизация (Lemmatization).

Объяснить, почему чрезмерная предварительная обработка может отрицательно влиять на словарные методы и Transformer-модели, особенно при удалении отрицаний или других признаков эмоциональной окраски.

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english")) - {"not", "no", "nor", "never", "n't"}

def light_preprocess(text):
    return " ".join(word_tokenize(text.lower()))

def full_preprocess(text):
    #2
    text = text.lower()
    #3
    text = re.sub(r"http\S+|@\w+", " ", text)
    #1
    tokens = word_tokenize(text)
    #3
    tokens = [t for t in tokens if t.isalpha() or t == "n't"]
    #4
    tokens = [t for t in tokens if t not in stop_words]
    #5
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

for part in [train_df, val_df, test_df]:
    part["light_text"] = part["text"].apply(light_preprocess)
    part["full_text"] = part["text"].apply(full_preprocess)

example = train_df.iloc[0]
print("Исходный:", example["text"])
print("Минимальный:", example["light_text"])
print("Полный:", example["full_text"])

In [ ]:
y_train = train_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values

prep_f1 = {}
for column in ["light_text", "full_text"]:
    vectorizer = TfidfVectorizer()
    X_tr = vectorizer.fit_transform(train_df[column])
    X_v = vectorizer.transform(val_df[column])
    model = LogisticRegression(max_iter=2000, random_state=SEED).fit(X_tr, y_train)
    prep_f1[column] = f1_score(y_val, model.predict(X_v), average="macro")
    print(column, "-> словарь:", len(vectorizer.vocabulary_), "слов, macro-F1 на val:", round(prep_f1[column], 4))

text_column = max(prep_f1, key=prep_f1.get)
print("Для TF-IDF используем:", text_column)

**Почему сильная предобработка вредит VADER и Transformer.** Эти модели сами понимают сырой текст, и многие «мусорные» для TF-IDF детали для них — сигнал тональности:

- `not`, `never`, `n't` переворачивают смысл: «not good» после удаления стоп-слов станет «good»;
- VADER усиливает оценку за `!!!`, КАПС и смайлики — после нижнего регистра и удаления символов этого нет;
- Transformer обучен на обычных предложениях; лемматизация и удаление слов делают текст непохожим на то, что он видел.

Поэтому VADER и RoBERTa получают исходный `text`, а обработанный текст используем только для TF-IDF.

### Шаг 3. Анализ тональности на основе словаря

- Реализовать классификатор тональности с использованием VADER или подходящего словаря эмоционально окрашенных слов.
- Рассчитать оценки тональности (Sentiment Scores).
- Классифицировать тексты по трём категориям тональности.
- При необходимости настройки пороговых значений использовать только валидационную выборку.
- Оценить качество полученных предсказаний.

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

#1
sia = SentimentIntensityAnalyzer()

#2
for part in [val_df, test_df]:
    part["vader_score"] = part["text"].apply(lambda t: sia.polarity_scores(t)["compound"])

#3
def vader_label(score, low, high):
    if score <= low:
        return 0
    if score >= high:
        return 2
    return 1

#4
best_f1, low_t, high_t = 0, None, None
for low in np.round(np.arange(-0.50, 0.01, 0.05), 2):
    for high in np.round(np.arange(0.00, 0.51, 0.05), 2):
        pred = val_df["vader_score"].apply(lambda s: vader_label(s, low, high))
        f1 = f1_score(y_val, pred, average="macro")
        if f1 > best_f1:
            best_f1, low_t, high_t = f1, low, high

default_pred = val_df["vader_score"].apply(lambda s: vader_label(s, -0.05, 0.05))
print("Стандартные пороги ±0.05: macro-F1 на val =", round(f1_score(y_val, default_pred, average="macro"), 4))
print(f"Подобранные пороги: low = {low_t:.2f}, high = {high_t:.2f}, macro-F1 на val = {best_f1:.4f}")

#3
test_df["vader_pred"] = test_df["vader_score"].apply(lambda s: vader_label(s, low_t, high_t))

### Шаг 4. Модель машинного обучения

- Преобразовать тексты в числовые векторы с помощью TF-IDF.
- Обучить классификатор Logistic Regression или Multinomial Naive Bayes.
- Проверить не менее двух конфигураций гиперпараметров.
- Оценить модель на валидационной выборке и выбрать лучшую конфигурацию.

In [ ]:
#1
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_train = tfidf.fit_transform(train_df[text_column])
X_val = tfidf.transform(val_df[text_column])
X_test = tfidf.transform(test_df[text_column])

#3
configs = [
    {"C": 0.5},
    {"C": 1.0},
    {"C": 4.0},
]

#2 #4
best_model, best_val_f1 = None, 0
for cfg in configs:
    model = LogisticRegression(max_iter=2000, random_state=SEED, **cfg)
    model.fit(X_train, y_train)
    val_f1 = f1_score(y_val, model.predict(X_val), average="macro")
    print(cfg, "-> macro-F1 на val:", round(val_f1, 4))
    if val_f1 > best_val_f1:
        best_model, best_val_f1, best_cfg = model, val_f1, cfg

print("Лучшая конфигурация:", best_cfg)
test_df["lr_pred"] = best_model.predict(X_test)

### Шаг 5. Модель на основе Transformer

- Выбрать предварительно обученную Transformer-модель для анализа тональности из Hugging Face, поддерживающую язык выбранного датасета и три целевых класса.
- Применить модель к тестовой выборке без дополнительного дообучения.
- При необходимости привести выходные метки модели к меткам используемого набора данных.
- Сравнить результаты с традиционными подходами.

Дообучение Transformer-модели (Fine-tuning) является необязательным.

**Модель:** `cardiffnlp/twitter-roberta-base-sentiment-latest` — RoBERTa, обученная на твитах, классы negative / neutral / positive.

In [ ]:
#1
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

sentiment_model = pipeline(
    "sentiment-analysis",
    device=device,
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length=128,
)

#3
label_map = {"negative": 0, "neutral": 1, "positive": 2}

#2
outputs = sentiment_model(test_df["text"].tolist(), batch_size=32)
test_df["tr_pred"] = [label_map[o["label"]] for o in outputs]
print(outputs[:3])

del sentiment_model  # освобождаем память

### Шаг 6. Оценка качества моделей

Рассчитать следующие метрики:

- **Accuracy** — точность классификации.
- **Precision (macro)** — макроусреднённая точность положительных предсказаний.
- **Recall (macro)** — макроусреднённая полнота.
- **F1-score (macro)** — макроусреднённая F1-мера.
- **Confusion Matrix** — матрица ошибок классификации.

Построить сравнительную таблицу с результатами оценки всех трёх подходов.

In [ ]:
#1
from sklearn.metrics import precision_score, recall_score

def metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision (macro)": precision_score(y_true, y_pred, average="macro"),
        "recall (macro)": recall_score(y_true, y_pred, average="macro"),
        "F1 (macro)": f1_score(y_true, y_pred, average="macro"),
    }

summary = pd.DataFrame({
    "VADER": metrics(y_test, test_df["vader_pred"]),
    "TF-IDF + LR": metrics(y_test, test_df["lr_pred"]),
    "Transformer": metrics(y_test, test_df["tr_pred"]),
}).T.round(4)
summary

In [ ]:
#2
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name, column in zip(axes, ["VADER", "TF-IDF + LR", "Transformer"], ["vader_pred", "lr_pred", "tr_pred"]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, test_df[column], display_labels=["neg", "neu", "pos"], ax=ax, colorbar=False
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

**Результаты на тесте (450 твитов):**

| подход | accuracy | macro-F1 |
|---|---|---|
| VADER | 0,536 | 0,537 |
| TF-IDF + LR | 0,609 | 0,599 |
| Transformer (RoBERTa) | 0,718 | 0,718 |

- **Transformer лучше всех** — на 12 п.п. macro-F1 выше TF-IDF и на 18 п.п. выше VADER.
- **Предобработка.** Минимальная обработка (нижний регистр + токены) дала на валидации 0,609, а полная (без стоп-слов, с леммами) — 0,591. Для коротких твитов каждое слово важно, и удаление слов только вредит. Поэтому для TF-IDF взят минимальный вариант.
- **Гиперпараметры.** Из трёх значений C лучшим на валидации оказалось C = 1.0, хотя разница маленькая (0,617–0,620).
- **Пороги VADER.** Подбор на валидации (low = −0,05, high = 0,40) поднял macro-F1 с 0,546 до 0,558: VADER слишком легко ставит «позитив», поэтому порог для позитива пришлось поднять.
- **Матрицы ошибок.** По диагонали — верные ответы, вне диагонали — какие классы модель путает. Нейтральный твит без ярких слов легко принять за позитивный или негативный.

### Шаг 7. Анализ ошибок

- Найти не менее пяти неправильно классифицированных отзывов.
- Сравнить предсказания различных моделей.
- Объяснить возможные причины ошибок: отрицания, сарказм, многозначность слов и предметно-специфические выражения.
- Определить наиболее эффективный подход и обосновать вывод экспериментальными результатами.

In [ ]:
#1
names = {0: "neg", 1: "neu", 2: "pos"}
table = test_df[["text", "label", "vader_pred", "lr_pred", "tr_pred"]].copy()
for column in ["label", "vader_pred", "lr_pred", "tr_pred"]:
    table[column] = table[column].map(names)

#2
table["ошиблись"] = (
    (test_df["vader_pred"] != test_df["label"]).astype(int)
    + (test_df["lr_pred"] != test_df["label"]).astype(int)
    + (test_df["tr_pred"] != test_df["label"]).astype(int)
)
print("Твитов, где ошиблись все три модели:", (table["ошиблись"] == 3).sum())
print("Твитов, где ошибся только Transformer:", ((test_df["tr_pred"] != test_df["label"]) & (table["ошиблись"] == 1)).sum())

pd.set_option("display.max_colwidth", 120)
table[table["ошиблись"] >= 2].sample(n=8, random_state=SEED)

**Разбор ошибок** (примеры из таблицы выше):

- **Сарказм.** «CONGRATULATIONS SINGLE PARENTS! #Trump says MORE TAXES for YOU! #MAGA!» — по разметке нейтральный, VADER и RoBERTa видят «CONGRATULATIONS» и капс и ставят позитив, а TF-IDF — негатив. Без знания контекста сарказм не распознать.
- **Нет эмоциональных слов.** «The average social security benefit was $1,341 in 2016. The average rent... is $1,634» — по разметке негативный (жалоба на дороговизну), но в тексте одни числа. VADER ставит позитив, RoBERTa — нейтральный. Тональность здесь в выводе, который делает читатель.
- **Предметная лексика.** «...whether homeopathy is a viable treatment...» и «An early abortion... tiny blob of cells» — по разметке нейтральные, но слова вроде «abortion», «irrelevant» модели связывают с негативом.
- **Склеенные слова и хэштеги.** «Hillary ClintonDedicated her life tochildren...» — слова без пробелов TF-IDF не узнаёт и ставит нейтральный, RoBERTa справляется.
- **Цитаты.** «It is time we Georgians did not depend only on others...» — по разметке позитив, но ошиблись все три модели: VADER и RoBERTa ставят нейтральный, TF-IDF — негатив (слово «not»).

На 46 твитах ошиблись все три модели — часто это спорные случаи, где и разметка неоднозначна. Только Transformer ошибся на 14 твитах из 450.

**Ответ на исследовательский вопрос.** Да, на этом датасете предобученный Transformer даёт существенное улучшение: macro-F1 0,718 против 0,599 у TF-IDF + LR и 0,537 у VADER. RoBERTa обучена на миллионах твитов и понимает контекст, сленг, эмодзи и отрицания. VADER видит только отдельные слова из словаря. TF-IDF + LR обучалась всего на 2100 твитах. Ограничения Transformer: он медленнее и тяжелее, а на сарказме и спорных твитах ошибается так же, как остальные.

# Задание 2. Автоматическое суммирование текста

Сравниваем экстрактивный (TextRank) и абстрактивный (BART) подходы на пяти статьях об автоматическом обнаружении музыкального плагиата.

### Шаг 1. Сбор данных

- Выбрать пять тематически связанных документов, предпочтительно новостных статей, научных аннотаций или технических отчётов.
- Каждый документ должен содержать не менее 250 слов.
- Указать источник и количество слов для каждого документа.
- Подготовить краткое эталонное резюме объёмом 100–150 слов, написанное человеком, либо использовать датасет с подходящими эталонными резюме.

Эталонное резюме должно быть подготовлено независимо от автоматических методов, качество которых оценивается.

**Документы:** пять фрагментов научных статей об автоматическом обнаружении музыкального плагиата (`article1.txt` … `article5.txt`).

**Эталонное резюме:** `reference.txt`, написано по статьям до запуска TextRank и BART; черновик подготовлен с помощью ИИ-ассистента и сверен со статьями.

In [ ]:
#1
import os
import urllib.request

GITHUB ="https://raw.githubusercontent.com/DIas-code/AITU/main/Programing%20for%20data%20analysis/assignment3/"
for name in [f"article{i}.txt" for i in range(1, 6)] + ["reference.txt"]:
    if not os.path.exists(name):
        urllib.request.urlretrieve(GITHUB + name, name)

documents = []
for i in range(1, 6):
    with open(f"article{i}.txt", encoding="utf-8") as f:
        documents.append(f.read())

#4
with open("reference.txt", encoding="utf-8") as f:
    reference_summary = f.read().strip()

#2 #3
sources = [
    "статья о системе поиска плагиата по мелодии: PCM -> мелодия -> MIDI -> сравнение строк",
    "«Music Plagiarism Detection Using Melody Databases», с. 685",
    "студенческий проект Music Plagiarism Detector (признаки MFCC, энтропия, энергия)",
    "«Music Plagiarism Detector», NCDMC-2017 (введение и виды плагиата)",
    "продолжение статьи о видах плагиата: ритм и мелодия",
]
pd.DataFrame({
    "источник": sources,
    "слов": [len(d.split()) for d in documents],
}, index=[f"article{i}" for i in range(1, 6)])

In [ ]:
print("Эталонное резюме:", len(reference_summary.split()), "слов\n")
print(reference_summary)

### Шаг 2. Предварительная обработка текста

- Выполнить токенизацию на уровне предложений.
- Удалить лишние пробелы и символы форматирования.
- Применить подходящую предварительную обработку для расчёта значимости предложений.
- Сохранить исходные предложения для формирования итоговых резюме.

In [ ]:
#2
def clean(text):
    text = text.replace("\x02", "")
    text = re.sub(r"Music Plagiarism Detection Using Melody Databases \d+", " ", text)
    text = re.sub(r"National Conference On .*?\| Page", " ", text)
    text = re.sub(r"\[\d+\]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

#1 #3 #4
doc_sentences = []
for doc in documents:
    sentences = sent_tokenize(clean(doc))
    sentences = [s for s in sentences if len(s.split()) >= 6]
    sentences = list(dict.fromkeys(sentences))
    doc_sentences.append(sentences)

for i, sentences in enumerate(doc_sentences):
    print(f"article{i + 1}: {len(sentences)} предложений")

Удалено: остатки переносов из PDF (`\x02`), колонтитулы, ссылки на литературу `[12]`, лишние пробелы, обрывки короче 6 слов и повторы предложений. Исходные предложения сохраняются для резюме; для расчёта значимости они переводятся в TF-IDF с нижним регистром и без английских стоп-слов.

### Шаг 3. Экстрактивное суммирование

Реализовать экстрактивный подход с использованием оценки значимости предложений на основе TF-IDF или TextRank.

- Рассчитать показатели значимости предложений.
- Выбрать три наиболее значимых предложения из каждого документа.
- Объединить пятнадцать выбранных предложений в один набор кандидатов.
- Выбрать шесть предложений для итогового объединённого резюме.
- Сохранить исходные формулировки и расположить предложения в логичном, удобном для чтения порядке.

In [ ]:
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

#1
def textrank_scores(sentences):
    vectors = TfidfVectorizer(stop_words="english").fit_transform(sentences)
    similarity = cosine_similarity(vectors)
    np.fill_diagonal(similarity, 0)
    graph = nx.from_numpy_array(similarity)
    return nx.pagerank(graph)

#2 #3
candidates = []
for d, sentences in enumerate(doc_sentences):
    scores = textrank_scores(sentences)
    top3 = sorted(scores, key=scores.get, reverse=True)[:3]
    print(f"--- article{d + 1} ---")
    for i in sorted(top3):
        print("•", sentences[i])
        candidates.append((d, i, sentences[i]))

print("\nКандидатов:", len(candidates))

In [ ]:
#4
candidate_texts = [c[2] for c in candidates]
scores = textrank_scores(candidate_texts)
ranked = sorted(scores, key=scores.get, reverse=True)

top6 = []
for d in range(5):
    top6.append([k for k in ranked if candidates[k][0] == d][0])
top6.append([k for k in ranked if k not in top6][0])

#5
top6 = sorted(top6, key=lambda k: (candidates[k][0], candidates[k][1]))
extractive_summary = " ".join(candidate_texts[k] for k in top6)

print(extractive_summary)
print("\nСлов:", len(extractive_summary.split()))

### Шаг 4. Абстрактивное суммирование

Использовать предварительно обученную Transformer-модель, например BART или T5.

- Сгенерировать резюме объёмом примерно 100–150 слов.
- Учесть ограничения модели по длине входного текста, применив подходящую стратегию, например разбиение текста на фрагменты (Chunking) или иерархическое суммирование.
- Зафиксировать параметры генерации.
- Проверить, насколько корректно сформированное резюме отражает информацию исходных документов.

Обучать новую языковую модель с нуля не требуется.

**Модель:** `facebook/bart-large-cnn`, на вход до 1024 токенов. **Стратегия:** иерархическое суммирование — статьи режутся на куски до 400 слов, резюме кусков склеиваются и суммируются ещё раз.

In [ ]:
#1
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=0 if torch.cuda.is_available() else -1)

#3
CHUNK_PARAMS = dict(max_length=80, min_length=30, num_beams=4, no_repeat_ngram_size=3, do_sample=False)
FINAL_PARAMS = dict(max_length=200, min_length=140, num_beams=4, no_repeat_ngram_size=3, do_sample=False)

#2
def chunks(sentences, max_words=400):
    result, current = [], []
    for s in sentences:
        if current and len(" ".join(current + [s]).split()) > max_words:
            result.append(" ".join(current))
            current = []
        current.append(s)
    if result and len(" ".join(current).split()) < 100:
        result[-1] += " " + " ".join(current)
    else:
        result.append(" ".join(current))
    return result

partial = []
for d, doc in enumerate(documents):
    for piece in chunks(sent_tokenize(clean(doc))):
        partial.append(summarizer(piece, **CHUNK_PARAMS)[0]["summary_text"])
print("Кусков текста:", len(partial))

abstractive_summary = summarizer(" ".join(partial), **FINAL_PARAMS)[0]["summary_text"]
print(abstractive_summary)
print("\nСлов:", len(abstractive_summary.split()))

del summarizer  # освобождаем память

In [ ]:
#4
all_sources = " ".join(clean(d) for d in documents)
for name, summary in [("экстрактивное", extractive_summary), ("абстрактивное", abstractive_summary)]:
    tokens = set(re.findall(r"\b(?:[A-Z][a-z]+|\d+)\b", summary))
    new = sorted(t for t in tokens if t not in all_sources)
    print(name, "— нет в источниках:", new)

### Шаг 5. Количественная оценка качества

Оценить итоговые экстрактивное и абстрактивное резюме относительно одного и того же эталонного резюме с использованием:

- ROUGE-1.
- ROUGE-2.
- ROUGE-L.

Представить результаты в сравнительной таблице.

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

rows = {}
for name, summary in [("Экстрактивное (TextRank)", extractive_summary), ("Абстрактивное (BART)", abstractive_summary)]:
    result = scorer.score(reference_summary, summary)
    rows[name] = {metric: round(score.fmeasure, 4) for metric, score in result.items()}
    rows[name]["слов"] = len(summary.split())

pd.DataFrame(rows).T

**Результат:** экстрактивное резюме выше по всем трём метрикам — ROUGE-1 0,422 против 0,336, ROUGE-2 0,059 против 0,055, ROUGE-L 0,208 против 0,188. Но оно и длиннее (167 слов против 112), а ROUGE-1 растёт, когда в тексте больше слов из эталона. ROUGE-2 у обоих очень низкий: эталон написан своими словами, и пар слов подряд, совпадающих с ним, почти нет. ROUGE считает совпадения слов, а не смысл, поэтому нужен и качественный анализ.

### Шаг 6. Качественный анализ

Оценить оба резюме по следующим критериям:

- Полнота охвата информации.
- Связность и читаемость.
- Избыточность информации.
- Фактическая согласованность с исходными документами.
- Грамматическая корректность.

Определить не менее трёх различий между экстрактивным и абстрактивным суммированием. Рассмотреть как минимум один пример, в котором абстрактивная модель изменяет, пропускает или добавляет информацию.

| критерий | экстрактивное (TextRank) | абстрактивное (BART) |
|---|---|---|
| Полнота | есть все 5 статей (так построен отбор); нет отпечатков MFCC из статьи 3 — выбрано предложение про спектрограммы | статьи 1, 2/3 и 4: нет отпечатков MFCC (статья 3) и плагиата ритма (статья 5) |
| Связность и читаемость | предложения из разных статей не связаны между собой, есть заголовки внутри предложений («Sampling Plagiarism The term...») | читается ровнее, но это склейка фраз без связок |
| Избыточность | два почти одинаковых предложения из статьи 2 про поиск похожих мелодий | повторов нет |
| Фактическая точность | все предложения дословно из статей | числа взяты верно (2 млрд $ в Корее, 1,4 → 1,8 млрд треков), но смешаны разные статьи — см. пример ниже |
| Грамматика | исходная, но с ошибками самих статей («two music have», «litErature») | обрыв: «...more than 2 billion dollar (2009) The system receives...» — нет точки, два предложения склеены |

**Три различия между подходами:**

1. Экстрактивное резюме копирует предложения дословно. Абстрактивное генерирует текст само, хотя BART в основном тоже переписал куски почти без изменений.
2. Экстрактивное не может сократить или объединить предложение: в резюме попадают длинные предложения целиком, вместе с лишними частями. Абстрактивное может сокращать.
3. Экстрактивный метод выбирает то, что похоже на остальной текст (TextRank). Абстрактивный пишет про то, что идёт в начале кусков: BART склонен к началу текста, как в новостях.

**Пример, где абстрактивная модель меняет и пропускает информацию.** BART пишет: «With our plagiarism detection system, composers can easily search for the melodies similar to their ones». В общем резюме пяти статей это звучит как вывод про всю тему, хотя «our system» — система конкретных авторов (фраза из статьи 2, в статье 3 она повторена). Читатель не узнает, чья это система. Кроме того, BART полностью пропустил подход с отпечатками MFCC/энтропии/энергии (статья 3) и плагиат ритма — Amen Break (статья 5), хотя в эталоне это есть. Автоматическая проверка (слова с большой буквы и числа, которых нет в статьях) ничего не нашла — значит, новых фактов BART не выдумал. Ошибки здесь в пропусках и в смешивании источников.

**Ответ на исследовательский вопрос.** Ни один подход не идеален:
- **экстрактивный** лучше по ROUGE и по фактической точности — ничего не придумывает, — но читается хуже и содержит повторы;
- **абстрактивный** короче и глаже, но пропускает темы, может склеить предложения с ошибкой и смешать источники.

Для научных статей, где важна точность, лучший баланс даёт экстрактивный подход или гибрид: сначала выбрать предложения TextRank, потом дать модели их переформулировать и проверить результат по источникам.

# Задание 3. Жанры музыки по текстам песен

Жанры: Pop, Rock, Hip-Hop, Country.

### Шаг 1. Сбор данных

- Выбрать датасет, содержащий тексты песен и метки музыкальных жанров.
- Выбрать четыре жанра, например Pop, Rock, Hip-Hop и Country.
- Использовать не менее 100 песен каждого жанра (минимум 400 песен).
- Использовать один язык для всего корпуса, предпочтительно английский.
- Удалить дублирующиеся тексты песен.
- Проанализировать распределение длины текстов и количество песен по жанрам.

**Датасет:** `Yegor25/lyrics_genre_dataset_medium` (Hugging Face, 468 тыс. песен). Берём песни с одним жанром; `rap` и `hip-hop` → Hip-Hop.

In [ ]:
#1
from datasets import load_dataset

lyrics_raw = load_dataset("Yegor25/lyrics_genre_dataset_medium", split="train")
print("Всего песен:", len(lyrics_raw), "| колонки:", lyrics_raw.column_names)

#2
genre_map = {"pop": "Pop", "rock": "Rock", "rap": "Hip-Hop", "hip-hop": "Hip-Hop", "country": "Country"}

def our_genre(genres):
    return genre_map.get(genres[0].lower()) if len(genres) == 1 else None

#4
english = lyrics_raw.filter(lambda batch: [lang == "english" for lang in batch["language"]], batched=True)

parts = []
for name in ["Country", "Hip-Hop", "Pop", "Rock"]:
    part = english.filter(lambda batch: [our_genre(g) == name for g in batch["genre"]], batched=True)
    print(f"{name}: {len(part)} песен на английском")
    part = part.shuffle(seed=SEED).select(range(1000)).to_pandas()
    part["genre"] = name
    parts.append(part[["lyrics", "genre"]])
songs = pd.concat(parts, ignore_index=True)

#5
print("Дубликатов текста:", songs.duplicated(subset="lyrics").sum())
songs = songs.drop_duplicates(subset="lyrics")
songs = songs[songs["lyrics"].str.split().str.len() >= 50]

#3
songs = songs.groupby("genre").sample(n=300, random_state=SEED).reset_index(drop=True)
print(songs["genre"].value_counts())

In [ ]:
#6
songs["n_words"] = songs["lyrics"].str.split().str.len()
songs.groupby("genre")["n_words"].describe().round(0)

В каждом жанре ровно 300 песен — классы сбалансированы. Песни Hip-Hop примерно в два раза длиннее остальных: медиана 408 слов против 188–218 у Country, Pop и Rock. Поэтому сравнивать жанры по абсолютным частотам нельзя — нужны относительные (Шаги 3 и 4).

### Шаг 2. Предварительная обработка текста

Реализовать:

- Приведение текста к нижнему регистру.
- Токенизацию.
- Удаление знаков препинания и ненужных символов.
- Удаление стоп-слов.
- Лемматизацию.

Объяснить влияние предварительной обработки на лексические характеристики текстов песен.

In [ ]:
stop_words_en = set(stopwords.words("english"))

def preprocess_lyrics(text):
    #1
    text = text.lower()
    #3
    text = re.sub(r"\[.*?\]", " ", text)
    #2
    tokens = word_tokenize(text)
    #3
    tokens = [t for t in tokens if t.isalpha()]
    #4
    tokens = [t for t in tokens if t not in stop_words_en and len(t) > 2]
    #5
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

songs["tokens"] = songs["lyrics"].apply(preprocess_lyrics)
songs["clean"] = songs["tokens"].apply(" ".join)

raw_words = songs["lyrics"].str.lower().str.split()
pd.DataFrame({
    "до обработки": [raw_words.str.len().mean().round(0), len(set(w for ws in raw_words for w in ws))],
    "после обработки": [songs["tokens"].str.len().mean().round(0), len(set(w for ws in songs["tokens"] for w in ws))],
}, index=["слов в песне (среднее)", "разных слов во всём корпусе"])

После обработки в средней песне остаётся 131 слово из 276 (−53%), а словарь всего корпуса сокращается с 22 827 до 13 567 разных слов (−41%).

- Ушли служебные слова (I, you, the, and) — они есть в каждой песне и жанр не различают.
- Формы слов объединились: `eyes` → `eye`, `hands` → `hand`.
- Но вместе с этим ушли и признаки стиля: повторы, междометия короче трёх букв (`oh`, `ah`), апострофы разговорной речи (`lovin'` → `lovin`).
- Лемматизатор без указания части речи меняет только существительные, поэтому `loved` и `love` остались разными словами.

Для анализа тематики такая обработка подходит, для анализа стиля исполнения — нет.

### Шаг 3. Частотный анализ

Для каждого жанра:

- Определить 10 наиболее часто встречающихся слов.
- Рассчитать их абсолютные и относительные частоты.
- Визуализировать результаты с помощью столбчатых диаграмм.
- Сравнить особенности словарного состава различных жанров.

In [ ]:
from collections import Counter

GENRES = ["Country", "Hip-Hop", "Pop", "Rock"]

#1 #2
genre_counts = {}
for g in GENRES:
    genre_counts[g] = Counter(w for tokens in songs.loc[songs["genre"] == g, "tokens"] for w in tokens)

top10 = {}
for g in GENRES:
    total = sum(genre_counts[g].values())
    top10[g] = pd.DataFrame(genre_counts[g].most_common(10), columns=["слово", "абс."])
    top10[g]["отн., %"] = (top10[g]["абс."] / total * 100).round(2)
    print(f"--- {g} (всего слов: {total}) ---")
    print(top10[g].to_string(index=False), "\n")

In [ ]:
#3
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, g in zip(axes.flatten(), GENRES):
    table = top10[g][::-1]
    ax.barh(table["слово"], table["абс."])
    ax.set_title(g)
    ax.set_xlabel("сколько раз встречается")
plt.tight_layout()
plt.show()

**Сравнение жанров.**

- Общее ядро у всех жанров: `like`, `know`, `got`, `love`, `time` — песни в целом про чувства и время.
- **Hip-Hop** заметно отличается: ненормативная лексика и сленг (`nigga`, `shit`, `bitch`, `fuck`), `yeah`, `back`. Слова `love` нет в его топ-10, а в остальных жанрах оно на 1–3 месте.
- **Country:** `little`, `heart`, `gon` (от gonna).
- **Pop:** `say`, `feel`, `never`.
- **Rock:** `come`, `want`, `let`, `never`.

Pop и Rock по топ-10 почти не различаются — это объясняет, почему классификатор их путает.

### Шаг 4. Анализ лексических категорий

Создать словари минимум для трёх лексических категорий, например:

- Временные выражения: morning, evening, night.
- Эмоции: love, sadness, happiness, anger.
- Части тела: heart, eyes, hands.

Рассчитать частоты слов каждой категории для каждого жанра.

Нормализовать частоты категорий по общему количеству слов соответствующего жанра, чтобы обеспечить корректность сравнения.

In [ ]:
#1
categories = {
    "время": {"morning", "evening", "night", "day", "today", "tonight", "tomorrow", "yesterday", "time"},
    "эмоции": {"love", "hate", "happy", "sad", "cry", "pain", "fear", "anger", "lonely", "joy"},
    "тело": {"heart", "eye", "hand", "head", "face", "body", "lip", "arm", "skin"},
}

#2 #3
rows = []
for g in GENRES:
    total = sum(genre_counts[g].values())
    for name, words in categories.items():
        count = sum(genre_counts[g][w] for w in words)
        rows.append({"жанр": g, "категория": name, "слов": count, "на 1000 слов": round(count / total * 1000, 1)})

category_df = pd.DataFrame(rows)
category_df.pivot(index="жанр", columns="категория", values="на 1000 слов")

Частоты на 1000 слов жанра:
- **Hip-Hop** — самый низкий во всех трёх категориях (время 11,9, тело 10,8, эмоции 9,3): словарь рэпа шире и разнообразнее, а эти «песенные» слова встречаются реже.
- **Время** чаще всего в Country (23,3) — истории про день, ночь, «старые времена».
- **Части тела** — в Rock (20,4).
- **Эмоции** почти одинаково в Rock (18,6) и Pop (18,3).

Без нормализации Hip-Hop выглядел бы лидером просто потому, что его песни вдвое длиннее.

### Шаг 5. Редкие и характерные слова

- Найти слова, встречающиеся не более трёх раз в пределах каждого жанра.
- Рассчитать показатели TF-IDF для определения характерных слов различных жанров.
- Сравнить редкие слова со словами, имеющими высокие значения TF-IDF.
- Объяснить, почему редкие слова не всегда являются наиболее информативными признаками для определения музыкального жанра.

In [ ]:
#1
rare_words = {}
for g in GENRES:
    rare_words[g] = [w for w, c in genre_counts[g].items() if c <= 3]
    share = len(rare_words[g]) / len(genre_counts[g]) * 100
    print(f"{g}: редких слов {len(rare_words[g])} ({share:.0f}% словаря), например: {rare_words[g][:8]}")

In [ ]:
#2
song_tfidf = TfidfVectorizer(min_df=5)
matrix = song_tfidf.fit_transform(songs["clean"])
terms = song_tfidf.get_feature_names_out()

for g in GENRES:
    in_genre = (songs["genre"] == g).values
    mean_in = np.asarray(matrix[in_genre].mean(axis=0)).ravel()
    mean_out = np.asarray(matrix[~in_genre].mean(axis=0)).ravel()
    top = [terms[j] for j in (mean_in - mean_out).argsort()[::-1][:10]]
    #3
    in_rare = [w for w in top if w in rare_words[g]]
    print(f"{g}: {top}")
    print(f"   из них редких (<= 3 раз в жанре): {in_rare}")

**Редкие и характерные слова.**

- Редкие слова (встречаются не больше 3 раз в жанре) — это 68–73% словаря каждого жанра. Примеры: `blacktop`, `enchantment`, `antoinette`, `speck`: случайные слова, имена, опечатки, единичные формы.
- Ни одно из 10 характерных слов по TF-IDF не редкое. TF-IDF выделяет слова, которые встречаются часто именно в этом жанре: у Hip-Hop — сленг и `money`, у Rock — `hell`, `kill`, `sun`, `tonight`, у Pop — `party`, `hey`, `yes`, у Country — `lord`, `old`, `blue`, `little`.

**Почему редкие слова неинформативны.** Слово, встреченное 1–3 раза, почти наверняка не встретится в новой песне этого жанра. Модель не может на него опереться, а часто это вообще имя или опечатка. Информативно слово, которое стабильно частое в одном жанре и редкое в других — это и меряет TF-IDF.

**Проблема в данных.** У Country в характерных словах `que`, `pra`, `amor`, `voc`, `vai` — это португальский и испанский. Часть песен в датасете ошибочно помечена как английские (вероятно, бразильский сертанежу, который помечен как Country). Это ошибка разметки, которая мешает классификации.

### Шаг 6. Автоматическая классификация жанров

Разработать два подхода:

**Модель 1:** TF-IDF + Multinomial Naive Bayes.

**Модель 2:** TF-IDF + Logistic Regression.

- Разделить данные на обучающую (80%) и тестовую (20%) выборки с использованием стратификации.
- Обучать TF-IDF-преобразователь только на обучающей выборке.
- При наличии информации об исполнителях рассмотреть разделение данных по исполнителям для исследования потенциальной утечки данных.
- Обучить обе классификационные модели.

**Исполнители:** в датасете нет колонки с исполнителем, поэтому разделение по исполнителям невозможно. Песни одного артиста могут попасть и в train, и в test, что может завысить качество.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

#1
X_train_text, X_test_text, y_train_genre, y_test_genre = train_test_split(
    songs["clean"], songs["genre"], test_size=0.20, stratify=songs["genre"], random_state=SEED
)

#2
lyrics_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9)
X_train_genre = lyrics_tfidf.fit_transform(X_train_text)
X_test_genre = lyrics_tfidf.transform(X_test_text)

#4
nb_model = MultinomialNB().fit(X_train_genre, y_train_genre)
lr_genre = LogisticRegression(max_iter=2000, random_state=SEED).fit(X_train_genre, y_train_genre)
print("train:", X_train_genre.shape[0], "песен | test:", X_test_genre.shape[0], "песен | признаков:", X_train_genre.shape[1])

### Шаг 7. Оценка и сравнение результатов

Рассчитать:

- Accuracy.
- Macro F1-score.
- Confusion Matrix.

Сравнить эффективность двух моделей.

Проанализировать не менее пяти ошибочных предсказаний и объяснить, какие музыкальные жанры чаще всего путаются.

Обсудить, достаточно ли только текстов песен для определения музыкального жанра, учитывая такие музыкальные характеристики, как ритм, инструменты и особенности вокала.

In [ ]:
#1
genre_preds = {"Naive Bayes": nb_model.predict(X_test_genre), "Logistic Regression": lr_genre.predict(X_test_genre)}

pd.DataFrame({
    name: {"accuracy": accuracy_score(y_test_genre, pred), "F1 (macro)": f1_score(y_test_genre, pred, average="macro")}
    for name, pred in genre_preds.items()
}).T.round(4)

In [ ]:
#2
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, pred) in zip(axes, genre_preds.items()):
    ConfusionMatrixDisplay.from_predictions(y_test_genre, pred, labels=GENRES, ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
#3
best_name = max(genre_preds, key=lambda n: f1_score(y_test_genre, genre_preds[n], average="macro"))
errors = pd.DataFrame({"верно": y_test_genre.values, "предсказано": genre_preds[best_name]}, index=X_test_text.index)
errors = errors[errors["верно"] != errors["предсказано"]]
print("Лучшая модель:", best_name, "| ошибок:", len(errors))
print(errors.groupby(["верно", "предсказано"]).size().sort_values(ascending=False).head(5), "\n")

for i in errors.sample(n=5, random_state=SEED).index:
    print(f"верно: {errors.loc[i, 'верно']} | предсказано: {errors.loc[i, 'предсказано']}")
    print(songs.loc[i, "lyrics"][:250].replace("\n", " / "), "...\n")

**Сравнение моделей:** Logistic Regression лучше Naive Bayes — accuracy 0,525 против 0,483, macro-F1 0,525 против 0,463. Это вдвое лучше случайного угадывания (0,25 на 4 жанра), но до уверенной классификации далеко.

**Что путается чаще всего** (по матрице ошибок LR):
- Rock ↔ Pop — по 19 ошибок в каждую сторону;
- Country → Rock (14), Pop → Country (14);

**Пять ошибок выше:**
- поп-песня с текстом о битвах и «inverted cross» (по словам это скорее метал) → Rock;
- поп-песня на смеси йоруба и английского → Hip-Hop;
- рок-песня «I'm a dyin' man... sugar... baby» → Country (разговорные формы, как в кантри);
- две рок-песни о любви и расставании → Pop.

Тексты о любви и отношениях похожи во всех жанрах.

**Достаточно ли одного текста?** Нет. Жанр в первую очередь определяется музыкой: ритм, темп, инструменты (гитарный перегруз, банджо, биты), манера вокала (рэп-читка). Одну и ту же песню о любви можно исполнить как поп, рок или кантри. Hip-Hop путается реже остальных: его нет среди пяти самых частых пар ошибок. Для остальных нужны аудиопризнаки: MFCC, темп, спектр.

**Ответ на исследовательский вопрос.** Лексические признаки и TF-IDF различают жанры лишь частично: macro-F1 0,52 при четырёх классах. Сильнее всего влияют:
1. сленг и ненормативная лексика — выделяют Hip-Hop;
2. тематические слова — `lord`, `old`, `little` у Country, `hell`, `kill` у Rock;
3. длина текста.

Pop, Rock и Country делят общую лексику о любви и времени, поэтому по тексту их различить трудно.

# Задание 4. Определение языка текста

8 языков: казахский, русский, английский, испанский, португальский, турецкий, украинский, немецкий. Близкородственные пары: испанский–португальский и русский–украинский.

### Шаг 1. Подготовка набора данных

- Использовать готовый многоязычный датасет, например Tatoeba или WiLI-2018.
- Выбрать от шести до восьми языков, включая казахский, русский и английский.
- Добавить как минимум одну пару близкородственных языков, например испанский и португальский.
- Собрать не менее 500 текстовых примеров для каждого языка.
- Удалить полные дубликаты.
- Разделить датасет на обучающую (80%) и тестовую (20%) выборки.

**Датасет:** WiLI-2018 (Thoma, 2018; zenodo.org/record/841984) — абзацы из Википедии, по 1000 на язык.

In [ ]:
#1
import os
import io
import zipfile
import urllib.request

if not os.path.exists("wili-2018/x_train.txt"):
    url = "https://zenodo.org/record/841984/files/wili-2018.zip"
    with urllib.request.urlopen(url) as response:
        zipfile.ZipFile(io.BytesIO(response.read())).extractall("wili-2018")

#2 #3
LANGS = {"kaz": "kk", "rus": "ru", "eng": "en", "spa": "es", "por": "pt", "tur": "tr", "ukr": "uk", "deu": "de"}

rows = []
for split in ["train", "test"]:
    with open(f"wili-2018/x_{split}.txt", encoding="utf-8") as f:
        texts = f.read().splitlines()
    with open(f"wili-2018/y_{split}.txt", encoding="utf-8") as f:
        labels = f.read().splitlines()
    rows.extend(zip(texts, labels))

lang_df = pd.DataFrame(rows, columns=["text", "iso"])
lang_df = lang_df[lang_df["iso"].isin(LANGS)].copy()
lang_df["lang"] = lang_df["iso"].map(LANGS)

#5
print("Дубликатов:", lang_df.duplicated(subset="text").sum())
lang_df = lang_df[["text", "lang"]].drop_duplicates(subset="text").reset_index(drop=True)
#4
print("Всего текстов:", len(lang_df))
print(lang_df["lang"].value_counts())

In [ ]:
#6
train_lang, test_lang = train_test_split(lang_df, test_size=0.20, stratify=lang_df["lang"], random_state=SEED)
train_lang = train_lang.copy()
test_lang = test_lang.copy()
print("train:", len(train_lang), "| test:", len(test_lang))

### Шаг 2. Эксперимент с предварительной обработкой

Сравнить две стратегии предварительной обработки.

**Подход A — Интенсивная предварительная обработка:**

- Привести тексты к нижнему регистру.
- Удалить цифры.
- Удалить знаки препинания и специальные символы.

**Подход B — Минимальная предварительная обработка:**

- Сохранить символы, характерные для отдельных языков.
- Удалить только лишние пробелы и артефакты форматирования.

Объяснить влияние предварительной обработки на распознавание казахского и других языков, содержащих специфические символы.

> **Важно:** оба подхода должны сохранять допустимые буквы Unicode. Не допускается удаление специфических букв казахского языка, таких как ə, ғ, қ, ң, ө, ұ, ү, һ и і.

In [ ]:
#A
def preprocess_A(text):
    text = text.lower()
    text = "".join(c if c.isalpha() or c.isspace() else " " for c in text)
    return re.sub(r"\s+", " ", text).strip()

#B
def preprocess_B(text):
    return re.sub(r"\s+", " ", text).strip()

for part in [train_lang, test_lang]:
    part["text_A"] = part["text"].apply(preprocess_A)
    part["text_B"] = part["text"].apply(preprocess_B)

kazakh_letters = set("әғқңөұүһі")
kk = train_lang[train_lang["lang"] == "kk"]
count = lambda texts: sum(c in kazakh_letters for t in texts for c in t.lower())
print("Казахских букв до обработки:", count(kk["text"]), "| после A:", count(kk["text_A"]), "| после B:", count(kk["text_B"]))
print("\nПример (A):", kk["text_A"].iloc[0][:150])

`c.isalpha()` в Python понимает буквы всех алфавитов, поэтому ә, ғ, қ, ң, ө, ұ, ү, һ, і сохраняются в обоих подходах — число казахских букв до и после совпадает. Если бы вместо этого чистили шаблоном `[^a-z]`, казахский и русский текст исчез бы целиком. Именно эти буквы — главный признак, по которому казахский отличается от русского: алфавит почти общий, но ә, қ, ң, ө, ұ, ү, і есть только в казахском.

### Шаг 3. Представление текстовых признаков

1. TF-IDF на уровне слов (Word-level TF-IDF).
2. TF-IDF на уровне символов с использованием символьных n-грамм (1–3).

Сравнить эффективность обоих подходов для задачи идентификации языка.

### Шаг 4. Разработка моделей

- Multinomial Naive Bayes.
- Logistic Regression.

Сравнить обученные модели с предварительно обученной системой определения языка, например fastText lid.176.

Предварительно обученную модель необходимо оценивать на той же отложенной тестовой выборке.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

y_train_lang = train_lang["lang"].values
y_test_lang = test_lang["lang"].values

#Шаг 3
features = {
    "слова": dict(analyzer="word"),
    "символы 1-3": dict(analyzer="char_wb", ngram_range=(1, 3)),
}

#Шаг 4
rows = []
trained = {}
for prep in ["A", "B"]:
    for feat_name, params in features.items():
        vectorizer = TfidfVectorizer(**params)
        X_tr = vectorizer.fit_transform(train_lang["text_" + prep])
        X_te = vectorizer.transform(test_lang["text_" + prep])
        for model_name, model in [("Naive Bayes", MultinomialNB()), ("Log. Regression", LogisticRegression(max_iter=2000, random_state=SEED))]:
            model.fit(X_tr, y_train_lang)
            pred = model.predict(X_te)
            rows.append({
                "обработка": prep, "признаки": feat_name, "модель": model_name,
                "accuracy": accuracy_score(y_test_lang, pred), "F1 (macro)": f1_score(y_test_lang, pred, average="macro"),
            })
            trained[(prep, feat_name, model_name)] = (vectorizer, model, pred)

lang_results = pd.DataFrame(rows).round(4).sort_values("F1 (macro)", ascending=False)
lang_results

**Предобученная модель:** fastText lid.176 (176 языков), на той же тестовой выборке.

In [ ]:
import fasttext

if not os.path.exists("lid.176.bin"):
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin", "lid.176.bin")

ft_model = fasttext.load_model("lid.176.bin")

def fasttext_predict(text):
    labels, scores = ft_model.predict(text.replace("\n", " "))
    return labels[0].replace("__label__", "")

ft_pred = test_lang["text"].apply(fasttext_predict).values
print("fastText: accuracy =", round(accuracy_score(y_test_lang, ft_pred), 4),
      "| F1 (macro) =", round(f1_score(y_test_lang, ft_pred, average="macro", labels=sorted(LANGS.values())), 4))
other = pd.Series([p for p in ft_pred if p not in LANGS.values()], dtype=str)
print("Ответы не из наших 8 языков:", other.value_counts().to_dict())

### Шаг 5. Оценка качества моделей

Рассчитать:

- Accuracy.
- Macro F1-score.
- Confusion Matrix.

Сравнить результаты для различных стратегий предварительной обработки, способов представления признаков и моделей классификации.

Определить наиболее эффективную конфигурацию.

In [ ]:
#3
best_row = lang_results.iloc[0]
best_key = (best_row["обработка"], best_row["признаки"], best_row["модель"])
best_vectorizer, best_lang_model, best_pred = trained[best_key]
print("Лучшая конфигурация:", best_key)

lang_order = sorted(LANGS.values())
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test_lang, best_pred, labels=lang_order, ax=ax, colorbar=False)
ax.set_title(" + ".join(best_key))
plt.show()

**Сравнение конфигураций.** На полных абзацах все 8 вариантов дают 98,2–98,8% — разница меньше 1 п.п.

- Лучшая конфигурация — слова + Logistic Regression + интенсивная обработка A: accuracy 0,988, macro-F1 0,988.
- Logistic Regression везде немного лучше Naive Bayes.
- Обработка A или B почти ничего не меняет. На длинном тексте язык и так виден по десяткам слов, а казахские буквы сохраняются в обоих вариантах.
- fastText lid.176 без всякого обучения на наших данных даёт 0,986 / 0,987 — столько же, сколько наши модели. При этом он выбирает из 176 языков и всего 3 раза ответил языком не из нашего списка.

Матрица ошибок: почти все ошибки уходят в `en` — разбираем в Шаге 7.

### Шаг 6. Тестирование устойчивости модели

#### Эксперимент A — Короткие тексты

Сравнить качество определения языка для текстов различной длины:

- 1–3 слова.
- 4–10 слов.
- Более 10 слов.

Короткие тексты — первые 1–3 и 4–10 слов тестовых абзацев.

In [ ]:
rng = random.Random(SEED)

def first_words(text, low, high):
    words = text.split()
    return " ".join(words[:rng.randint(low, high)])

length_sets = {
    "1-3 слова": test_lang["text_B"].apply(lambda t: first_words(t, 1, 3)),
    "4-10 слов": test_lang["text_B"].apply(lambda t: first_words(t, 4, 10)),
    "больше 10": test_lang["text_B"],
}

rows = []
for length, texts in length_sets.items():
    row = {"длина": length}
    for feat_name in ["слова", "символы 1-3"]:
        vectorizer, model, _ = trained[("B", feat_name, "Log. Regression")]
        row["LR, " + feat_name] = accuracy_score(y_test_lang, model.predict(vectorizer.transform(texts)))
    row["fastText"] = accuracy_score(y_test_lang, texts.apply(fasttext_predict))
    rows.append(row)

pd.DataFrame(rows).set_index("длина").round(4)

#### Эксперимент B — Тексты с ошибками

Создать модифицированную версию тестовой выборки, добавив контролируемые орфографические ошибки, например:

- Удаление символов.
- Вставка дополнительных символов.
- Замена символов.

Сравнить качество классификации до и после внесения ошибок.

Опечатки делаются буквами того же текста.

In [ ]:
def add_typos(text, p, rng):
    letters = [c for c in text if c.isalpha()] or ["a"]
    result = []
    for c in text:
        r = rng.random()
        if r < p:  #1
            continue
        elif r < 2 * p:  #2
            result.append(c)
            result.append(rng.choice(letters))
        elif r < 3 * p and c.isalpha():  #3
            result.append(rng.choice(letters))
        else:
            result.append(c)
    return "".join(result)

prep, feat_name, model_name = best_key
rows = []
for p in [0.0, 0.05, 0.10, 0.20]:
    noise_rng = random.Random(SEED)
    noisy = test_lang["text"].apply(lambda t: add_typos(t, p, noise_rng))
    noisy = noisy.apply(preprocess_A if prep == "A" else preprocess_B)
    pred = best_lang_model.predict(best_vectorizer.transform(noisy))
    rows.append({"ошибок на символ": f"{p * 3:.0%}", "accuracy": accuracy_score(y_test_lang, pred), "F1 (macro)": f1_score(y_test_lang, pred, average="macro")})

print("Пример с 30% ошибок:", add_typos(test_lang["text"].iloc[0], 0.10, random.Random(SEED))[:120])
pd.DataFrame(rows).round(4)

#### Эксперимент C — Смешение языков (Code-Switching)

- Составить не менее 30 предложений со смешением казахского и русского языков.
- Проанализировать предсказания обученной модели для смешанных текстов.
- Объяснить, почему классификатор с одной языковой меткой может некорректно обрабатывать смешение языков.

In [ ]:
#1
mixed = [
    "Сәлеметсіз бе, как дела?",
    "Мен сағат беске кофе ішемін, потом иду на работу.",
    "Бұл есеп очень сложный болды.",
    "Таңертең жүгірдім, а вечером отдыхал.",
    "Қаладағы движение қатты интенсивный.",
    "Маған кофе қажет, срочно.",
    "Ол концертке барды, но мне не понравилось.",
    "Бұл кітап интересная, прочитал за ночь.",
    "Жаңалықтарды қарадым, ничего нового.",
    "Концерт прошёл сәтті.",
    "Апам пирог жасады, вкусно получилось.",
    "Университетте лекция болды, по математике.",
    "Қалай ойлайсың, стоит ли идти?",
    "Мен бүгін тойдым, не могу больше есть.",
    "Ол машинамен келді, прямо к подъезду.",
    "Жаңа фильм көрдім, супер.",
    "Қазір сөйлесе алмаймын, перезвоню.",
    "Жұмыстан кейін кездесеміз, в кафе.",
    "Ол ағылшынша говорит жақсы.",
    "Бұл есеп тяжёлый, нужно время.",
    "Жаңа үй сатып алдым, ипотека на 15 лет.",
    "Мен бүгін тойға барамын, в семь вечера.",
    "Апам пирог пісірді, с яблоками.",
    "Жаңа жоба басталды, срочный.",
    "Университет жабық, карантин жарияланды.",
    "Концерт жақсы болды, зал был полный.",
    "Жаңа планшет сатып алдым, для работы.",
    "Жаңбыр жауды, промок до нитки.",
    "Мен бүгін ертерек тұрдым, в шесть утра.",
    "Жаңа курс басталды, по программированию.",
]
print("Предложений:", len(mixed))

#2
prep_fn = preprocess_A if prep == "A" else preprocess_B
X_mixed = best_vectorizer.transform([prep_fn(m) for m in mixed])
probabilities = best_lang_model.predict_proba(X_mixed)

mixed_df = pd.DataFrame({
    "предложение": mixed,
    "модель": best_lang_model.predict(X_mixed),
    "уверенность": probabilities.max(axis=1).round(2),
    "fastText": [fasttext_predict(m) for m in mixed],
})
print("Наша модель:", mixed_df["модель"].value_counts().to_dict())
print("fastText:", mixed_df["fastText"].value_counts().to_dict())
mixed_df

**A. Длина текста** (accuracy):

| длина | LR, слова | LR, символы 1–3 | fastText |
|---|---|---|---|
| 1–3 слова | 0,596 | 0,697 | 0,765 |
| 4–10 слов | 0,871 | 0,924 | 0,949 |
| больше 10 | 0,987 | 0,987 | 0,986 |

- Чем короче текст, тем хуже все модели.
- На коротких текстах символьные признаки выигрывают у словесных на 5–10 п.п.
- Лучше всех на коротких текстах fastText: он тоже работает на символьных n-граммах, но обучен на намного большем объёме текстов.

**B. Опечатки.** При ~15% испорченных символов точность падает с 0,988 до 0,964, при ~30% — до 0,890, при ~60% — до 0,474. Модель на словах чувствительна к опечаткам: испорченное слово — это новое слово, которого нет в словаре модели.

**C. Смешанный казахско-русский текст** (30 предложений).

- **Наша модель:** русский 18, казахский 9, украинский 3. Уверенность почти везде низкая (0,21–0,66): модель видит признаки двух языков сразу.
- **fastText:** русский 13, казахский 11, украинский 3, киргизский 2, татарский 1. Киргизский и татарский — тюркские языки, родственные казахскому, на кириллице.
- Правильного ответа здесь нет: в каждом предложении два языка. Классификатор с одной меткой обязан выбрать один язык и выбирает тот, чьих признаков больше. Например, «в семь вечера» перевешивает «Мен бүгін».
- Модели не видели смешанных примеров при обучении. Поэтому появляются «третьи» языки, похожие на оба: украинский (похож на русский), киргизский и татарский (похожи на казахский).

### Шаг 7. Анализ ошибок

- Найти не менее пяти неправильных предсказаний.
- Проанализировать влияние длины текста, общей лексики, орфографических ошибок и близкородственных языков.
- Объяснить, почему символьные признаки могут превосходить признаки на уровне слов при обработке коротких многоязычных текстов.
- Предложить способы повышения надёжности систем автоматического определения языка.

In [ ]:
#1
full_errors = test_lang[best_pred != y_test_lang][["text", "lang"]].copy()
full_errors["предсказано"] = best_pred[best_pred != y_test_lang]
print("Ошибок на полных текстах:", len(full_errors), "из", len(test_lang))
print(full_errors.assign(text=full_errors["text"].str[:120]).to_string(), "\n")

vectorizer, model, _ = trained[("B", "символы 1-3", "Log. Regression")]
short = length_sets["1-3 слова"]
short_pred = model.predict(vectorizer.transform(short))
short_errors = pd.DataFrame({"текст": short, "верно": y_test_lang, "предсказано": short_pred})
short_errors = short_errors[short_errors["верно"] != short_errors["предсказано"]]
print("Ошибок на текстах 1-3 слова:", len(short_errors), "из", len(short))
print("Частые пары (верно -> предсказано):")
print(short_errors.groupby(["верно", "предсказано"]).size().sort_values(ascending=False).head(6), "\n")
short_errors.sample(n=8, random_state=SEED)

Всего 19 ошибок на 1585 полных текстах. Из них:

1. **13 — ошибки в разметке датасета, а не в модели.** Абзац взят из Википедии на одном языке, но сам текст на другом:
   - 11 раз это английский текст — в основном библиографические ссылки внутри статей на португальском, русском, турецком (`Farrington, D. P. (1994b) Interactions between...`, `Thomson, Harry C. ...`); модель ответила `en` и по сути права;
   - один раз это немецкая ссылка в испанской статье;
   - один раз — русская ссылка, написанная латиницей.
2. **3 — близкородственные языки.** `es → pt` («Está diseñado para evaluar guías...», «Lugar de Realengo...») и `ru → uk` («Система католических школ Уиноны...»): общие слова и похожее написание.
3. **3 — списки имён и названий** (`Tim Heath, Murali Mohan Rao...`, `Maurice Herzog Sport Complex`): имена не принадлежат никакому языку.

**Короткие тексты.** Из 1–3 слов ошибочно 481 из 1585 (символьная LR). Частые пары: `en → tr` (55), `ru → uk` (52), `en → de` (35). Примеры: «Julie Benz», «Bussy-Saint-Georges», «Está» — одно имя или одно слово.

**Почему символьные признаки лучше на коротких текстах.** В тексте из 1–3 слов часто нет ни одного слова, которое модель видела при обучении. Тогда вектор на уровне слов пустой, и модель угадывает. Символьные n-граммы есть в любом слове, даже незнакомом: `қ`, `ің`, `ção`, `sch`, `ий` сразу выдают язык. Опечатка портит одно слово целиком, а из символьных n-грамм — только несколько.

**Как сделать определение языка надёжнее:**

- объединить словесные и символьные признаки;
- добавить в обучение короткие куски текста и тексты с опечатками;
- при низкой уверенности (например, меньше 0,5) отвечать «не знаю», а не угадывать;
- для смешанных текстов определять язык каждого слова или фрагмента, а не одну метку на всё предложение;
- почистить датасет от ссылок и списков имён.

**Ответ на исследовательский вопрос.** На длинных текстах выбор признаков и предобработки почти не важен — все варианты около 98–99%. Решает качество текста:

- короткий текст роняет точность до 60–77%;
- сильные опечатки — до 47%;
- для казахского и русского главное — сохранить специфические буквы (ә, қ, ң, ө, ұ, ү, і), и оба наших подхода их сохраняют;
- основная путаница — внутри близких пар ru–uk и es–pt, а для казахского — с другими тюркскими языками (киргизский, татарский у fastText);
- символьные n-граммы — самый устойчивый вариант для коротких и испорченных текстов.

## Итог

| задание | лучший подход | главный результат |
|---|---|---|
| 1. Тональность | Transformer (RoBERTa) | macro-F1 0,718 против 0,599 (TF-IDF + LR) и 0,537 (VADER) |
| 2. Суммирование | экстрактивный (TextRank) по ROUGE и точности | ROUGE-1 0,422 против 0,336; BART читается лучше, но пропускает темы |
| 3. Жанры песен | TF-IDF + Logistic Regression | macro-F1 0,525; Hip-Hop путается реже остальных жанров |
| 4. Язык текста | TF-IDF слов + LR на полных текстах; на коротких — символьные n-граммы и fastText | 98,8% на абзацах; на 1–3 словах 70% (символы) и 77% (fastText) |

**Воспроизводимость:** seed = 42 везде (выборки, деление, модели). Датасеты: TweetEval (test), article1–5.txt + reference.txt, `Yegor25/lyrics_genre_dataset_medium`, WiLI-2018. Модели: `cardiffnlp/twitter-roberta-base-sentiment-latest`, `facebook/bart-large-cnn`, fastText `lid.176.bin`. Утечки данных нет: дубликаты удалены до деления, TF-IDF обучается только на train, пороги VADER подобраны на валидации, RoBERTa не видела тестовых твитов при обучении.